# RAG-Powered Football Rules & Tactics Assistant

### End-to-end offline pipeline (Core Track)

This notebook builds the retrieval-augmented generation (RAG) pipeline behind a football
rules and tactics assistant. It is organised as a report:

1. **Load & Inspect** the raw documents
2. **Chunking Strategy** - fixed-size token windows with a written justification
3. **Embeddings & Vector Store** - sentence-transformers into a persistent ChromaDB store
4. **Retrieval & Prompting** - a `retrieve()` function and a grounded prompt for Ollama (`phi3:mini`)
5. **Evaluation** - 10 test questions, including deliberately off-topic ones
6. **Export** - persisted vector store + `config.json` for the backend

**Stack:** Python 3.12, sentence-transformers, ChromaDB, Ollama (`phi3:mini`).

## 1. Load & Inspect

Load every `.txt` (and `.md`) file from `data/raw_docs/`, report how many were loaded,
and capture any file that failed to load so the failure is visible rather than silent.

In [1]:
import os
from pathlib import Path
import pandas as pd

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

# Resolve paths whether the kernel starts in the repo root or in notebooks/
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DOCS = PROJECT_ROOT / "data" / "raw_docs"
VECTOR_STORE = PROJECT_ROOT / "data" / "vector_store"
VECTOR_STORE.mkdir(parents=True, exist_ok=True)

# Configuration for the whole pipeline (used again in section 6)
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
EMBED_MODEL_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
OLLAMA_MODEL = "phi3:mini"
COLLECTION_NAME = "football_docs"

print("Project root :", PROJECT_ROOT)
print("Raw docs dir :", RAW_DOCS)
print("Vector store :", VECTOR_STORE)


def load_documents(folder):
    """Read every .txt/.md file; return (documents, failures)."""
    documents, failed = [], []
    paths = sorted({p for pattern in ("*.txt", "*.md") for p in folder.glob(pattern)})
    for path in paths:
        try:
            text = path.read_text(encoding="utf-8").strip()
            if not text:
                failed.append((path.name, "file is empty"))
                continue
            documents.append({
                "source": path.name,
                "text": text,
                "chars": len(text),
                "words": len(text.split()),
                "lines": text.count("\n") + 1,
            })
        except Exception as exc:  # noqa: BLE001 - surface any read/decoding failure
            failed.append((path.name, f"{type(exc).__name__}: {exc}"))
    return documents, failed


documents, failed_files = load_documents(RAW_DOCS)
print(f"\nLoaded {len(documents)} documents, {len(failed_files)} failed to load.")
for name, reason in failed_files:
    print("  FAILED:", name, "-", reason)

stats = pd.DataFrame(documents)[["source", "chars", "words", "lines"]]
stats.loc["TOTAL"] = ["-", stats["chars"].sum(), stats["words"].sum(), stats["lines"].sum()]
stats

Project root : C:\Users\Abdallah ElBarawy\Desktop\ITI project\ITI-Level-2-Final-project
Raw docs dir : C:\Users\Abdallah ElBarawy\Desktop\ITI project\ITI-Level-2-Final-project\data\raw_docs
Vector store : C:\Users\Abdallah ElBarawy\Desktop\ITI project\ITI-Level-2-Final-project\data\vector_store

Loaded 12 documents, 0 failed to load.


,source,chars,words,lines
0,corners.txt,2299,403,9
1,false_nine.txt,2446,419,9
2,formation_4-3-3.txt,2345,361,9
3,formation_4-4-2.txt,2194,360,9
4,gegenpressing.txt,2284,365,9
5,laws_of_the_game_fouls_and_cards.txt,2901,464,9
6,offside_rule.txt,2667,463,9
7,offside_trap.txt,2219,378,9
8,pressing_and_counter_attacking.txt,2587,410,9
9,throw_ins.txt,2253,398,9


## 2. Chunking Strategy

**Choice: fixed-size chunks of 500 tokens with 50 tokens of overlap (stride = 450).**

Each source file is a short, single-topic article of roughly 2-3 KB, so a simple fixed
window is the right cost/benefit point: it is deterministic, trivial to reproduce, and
cheap to store - all of which matter on a tight timeline.

**Why 500 tokens?** A 500-token window holds a full paragraph or two, which matches the
granularity of the questions we expect (a rule, a definition, a role description). Smaller
chunks would fragment explanations such as the offside rule across many vectors and dilute
their meaning; larger chunks would mix several ideas into one vector and blunt retrieval
precision.

**Why 50 tokens of overlap?** Overlap protects against a fact being split across a chunk
boundary. If a key sentence straddles the end of one window and the start of the next, the
50-token overlap guarantees it still appears intact in at least one chunk.

**Model fit matters here.** The embedding model `sentence-transformers/multi-qa-MiniLM-L6-cos-v1`
accepts up to **512 tokens**, so 500-token chunks are embedded in full. (A common default such
as `all-MiniLM-L6-v2` caps at 256 tokens, which would silently truncate almost 40% of every
chunk and hide content from retrieval - a trap worth calling out.)

Token counts are measured with the **same tokenizer** as the embedding model, so the window
size is consistent with what the model actually sees.

In [2]:
from transformers import AutoTokenizer

from huggingface_hub import logging as hf_logging
hf_logging.set_verbosity_error()

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
# We control the window size ourselves, so stop the tokenizer warning about documents
# that are longer than the model's own 512-token limit.
tokenizer.model_max_length = 1_000_000
print("Tokenizer:", EMBED_MODEL_NAME, "| vocab:", tokenizer.vocab_size)


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping token windows, preserving the original characters."""
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]
    stride = chunk_size - overlap
    pieces = []
    for start in range(0, len(ids), stride):
        end = min(start + chunk_size, len(ids))
        if start >= end:
            break
        char_start = offsets[start][0]
        char_end = offsets[end - 1][1]
        piece = text[char_start:char_end].strip()
        if piece:
            pieces.append({"text": piece, "n_tokens": end - start})
        if end >= len(ids):
            break
    return pieces, len(ids)


chunks = []
for doc in documents:
    pieces, doc_tokens = chunk_text(doc["text"])
    for i, piece in enumerate(pieces):
        chunks.append({
            "id": f"{doc['source']}::chunk{i}",
            "source": doc["source"],
            "chunk_index": i,
            "text": piece["text"],
            "n_tokens": piece["n_tokens"],
            "doc_tokens": doc_tokens,
        })

chunk_stats = pd.DataFrame(
    [{k: c[k] for k in ("id", "source", "chunk_index", "n_tokens")} for c in chunks]
)
print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print("tokens/chunk -> min={}  max={}  mean={:.1f}".format(
    chunk_stats.n_tokens.min(), chunk_stats.n_tokens.max(), chunk_stats.n_tokens.mean()))
chunk_stats.groupby("source").size().rename("chunks").to_frame().T

Tokenizer: sentence-transformers/multi-qa-MiniLM-L6-cos-v1 | vocab: 30522
Created 15 chunks from 12 documents
tokens/chunk -> min=56  max=500  mean=401.3


source,corners.txt,false_nine.txt,formation_4-3-3.txt,formation_4-4-2.txt,gegenpressing.txt,laws_of_the_game_fouls_and_cards.txt,offside_rule.txt,offside_trap.txt,pressing_and_counter_attacking.txt,throw_ins.txt,tiki_taka.txt,total_football.txt
chunks,1,1,1,1,1,2,2,1,2,1,1,1


## 3. Embeddings & Vector Store

Each chunk is embedded with `sentence-transformers/multi-qa-MiniLM-L6-cos-v1` (384-dimensional,
trained for question-to-passage retrieval) and normalized to unit length so that cosine
similarity behaves as expected. Vectors, raw text and metadata are written to a **persistent**
ChromaDB collection stored in `data/vector_store/`; the collection uses `hnsw:space = cosine`.

The write is an idempotent `upsert`, so re-running the notebook replaces vectors instead of
duplicating them.

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL_NAME)
EMBED_DIM = embedder.get_embedding_dimension()
print("Model:", EMBED_MODEL_NAME)
print("Embedding dimension:", EMBED_DIM, "| max tokens:", embedder.max_seq_length)

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(
    texts, normalize_embeddings=True, batch_size=32, show_progress_bar=True
)

client = chromadb.PersistentClient(path=str(VECTOR_STORE))
try:
    client.delete_collection(COLLECTION_NAME)  # keep re-runs clean
except Exception:
    pass
collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=None,
    metadata={"hnsw:space": "cosine"},
)

collection.upsert(
    ids=[c["id"] for c in chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"], "n_tokens": c["n_tokens"]} for c in chunks],
)
print("\nCollection:", COLLECTION_NAME, "| stored vectors:", collection.count())

probe_vec = embedder.encode(["offside"], normalize_embeddings=True)[0].tolist()
probe = collection.query(query_embeddings=[probe_vec], n_results=3, include=["metadatas", "distances"])
print("Sanity query 'offside' ->",
      [(m["source"], round(d, 3)) for m, d in zip(probe["metadatas"][0], probe["distances"][0])])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Embedding dimension: 384 | max tokens: 512


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Collection: football_docs | stored vectors: 15
Sanity query 'offside' -> [('offside_rule.txt', 0.302), ('offside_rule.txt', 0.317), ('offside_trap.txt', 0.448)]
